# 05: Pipeline Test

Test the full pipeline: Safety -> Preprocessing -> RAG + KG -> Fusion -> Output

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
import sys
sys.path.append('/content/drive/MyDrive/mental_health_rag_kg')  # <-- ADD THIS LINE
import os
os.chdir('/content/drive/MyDrive/mental_health_rag_kg')


from src.pipeline import MentalHealthPipeline
pipeline = MentalHealthPipeline()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Test 1: Safe informational query
result = pipeline.run('What are the symptoms of anxiety and depression?', verbose=True)
print('=== FINAL OUTPUT ===')
print(result['final_response'])
print(f"Confidence: {result['final_confidence']}")

[SAFETY] Input is safe.
[PREPROC] Symptoms: ['anxiety', 'depression']
[PREPROC] Refined query: Here's a refined query:

"(Anxiety OR anxiety disorder) AND (Depression OR major depressive disorder) AND symptoms"

This query is designed to retrieve relevant information on the symptoms of both anxiety and depression, using specific medical terminology to ensure accurate results.
[RAG] Retrieving from vector store...
Loaded index with 100 vectors.
[RAG] Confidence: {'R_retrieval': 0.545, 'G_grounding': 0.625, 'C_consistency': 0.355, 'llm_confidence': 0.525}
[KG] Querying knowledge graph...
[KG] Confidence: {'K_known': 1.0, 'M_alignment': 0.24, 'G_graph': 0.7, 'kg_confidence': 0.712}
[FUSION] Strategy: cautious_merged
[FUSION] Hallucination score: 0.5
[FUSION] Final confidence: 0.712
=== FINAL OUTPUT ===
Based on available information:
Based on verified knowledge graph data:
- OCD: matches 1/4 symptoms (25.0%)
- Bipolar Disorder: matches 1/5 symptoms (20.0%)

This is not a diagnosis. Consul

In [6]:
# Test 2: Symptom description for matching
result = pipeline.run('I feel very sad, cannot sleep, and have no energy.', verbose=True)

[SAFETY] Input rejected.


In [7]:
# Test 3: Safety rejection
result = pipeline.run('I want to hurt myself and end my life.', verbose=True)
print('=== SAFETY OUTPUT ===')
print(result['final_response'])

[SAFETY] Input rejected.
=== SAFETY OUTPUT ===
[REJ] I cannot provide assistance for this request. If you are in crisis, please contact emergency services or a mental health helpline immediately. You matter, and help is available.



In [8]:
# Test 4: Low confidence / abstention test
result = pipeline.run('How many angels can dance on the head of a pin?', verbose=True)

[SAFETY] Input is safe.
[PREPROC] Symptoms: []
[PREPROC] Refined query: "Delusional thinking" OR "psychotic symptoms" OR "schizotypal personality disorder" 

Note: The user's input appears to be a metaphorical or philosophical question, which may be unrelated to their actual concern. However, the question itself can be an indicator of delusional thinking or disorganized thought patterns, which are common symptoms in certain mental health conditions. The refined query aims to capture relevant information related to these potential underlying issues.
[RAG] Retrieving from vector store...
[RAG] Confidence: {'R_retrieval': 0.527, 'G_grounding': 0.549, 'C_consistency': 0.272, 'llm_confidence': 0.471}
[KG] Querying knowledge graph...
[KG] Confidence: {'K_known': 0.0, 'M_alignment': 0.0, 'G_graph': 0.0, 'kg_confidence': 0.0}
[FUSION] Strategy: abstain
[FUSION] Hallucination score: 0.5
[FUSION] Final confidence: 0.235


In [9]:
# Interactive chat mode (optional)
pipeline.chat()

Mental Health RAG+KG Assistant
Type 'exit' to quit, 'verbose' to toggle details.

Bot: [REJ] I cannot provide assistance for this request. If you are in crisis, please contact emergency services or a mental health helpline immediately. You matter, and help is available.


Bot: Based on available information:
Based on verified knowledge graph data:
- OCD: matches 1/4 symptoms (25.0%)

This is not a diagnosis. Consult a professional.

Note: This answer has moderate confidence. Please verify with a professional.
[Confidence: 0.715]

Bot: [REJ] I cannot provide assistance for this request. If you are in crisis, please contact emergency services or a mental health helpline immediately. You matter, and help is available.


Bot: [REJ] I do not have sufficient reliable information to answer this safely. Please consult a qualified mental health professional for accurate guidance.

[Confidence: 0.45]

Bot: [REJ] I cannot provide assistance for this request. If you are in crisis, please contact e